# M7 Run 9 - transfer A/B (does pretraining help?)

Loads the Run 8 pretrained encoder and compares 3 modes on the mini-OOF (train folds 5-8, test 1-4, 3 seeds @5000): **scratch** (random init) / **frozen** (pretrained encoder frozen + head trained) / **finetune** (pretrained + full fine-tune). Config = frozen tuned (both + strong + base).

> **SUB-GATE (anti-zombie):** the best pretrained mode must beat `scratch` by the override criterion (per-seed >=3 sigma AND std not increasing). PASS -> Run 10 (full OOF pretrained = M7-v2). FAIL -> STOP, M7-v1 stays final.

> Reload Window first (clean memory); mmap keeps the peak ~6 GB.

### Block 9.0: Setup

In [ ]:
# M7 Run 9 - TRANSFER A/B: does the pretrained encoder (Run 8) help? 3 modes on the mini-OOF proxy:
#   scratch (random init) | frozen (pretrained encoder frozen + head trained) | finetune (pretrained + full fine-tune).
# Config = frozen tuned (both + strong aug + base reg @5000). SUB-GATE: pretrained must beat scratch by the override
# criterion (per-seed >=3 sigma AND std not increasing), else STOP (no Run 10).
import os, sys, json, time, warnings, shutil, zipfile
import numpy as np, pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score
warnings.filterwarnings('ignore')
ROOT=r'.'
PROCESSED=os.path.join(ROOT,'data','processed'); SRC=os.path.join(ROOT,'src')
METRICS=os.path.join(ROOT,'reports','metrics'); MODELS=os.path.join(ROOT,'models','M7_run9')
CACHE_DIR=os.path.join(PROCESSED,'m7_run2_cache'); ENC_PATH=os.path.join(ROOT,'models','M7_pretrain','encoder.pt')
for d in (METRICS,MODELS): os.makedirs(d,exist_ok=True)
sys.path.insert(0,SRC); from evaluation import _boot_ci
RANDOM_STATE=42; N_JOBS=6; SEEDS=[0,1,2]
EPOCHS=60; PATIENCE=10; BATCH=64; LR=1e-3; WD=1e-4; PDROP=0.3
FOCAL_GAMMA=2.0; FOCAL_ALPHA=0.5; VAL_FRAC=0.15; N_NEG_TRAIN=12000
TEST_FOLDS=[1,2,3,4]; TRAIN_FOLDS=[5,6,7,8]
MODES=['scratch','frozen','finetune']; BASELINE='scratch'
print('M7 Run 9 | transfer A/B %s | encoder=%s'%(MODES,os.path.exists(ENC_PATH)))
print('Block 9.0 OK.')


### Block 9.1: mmap cache + mini-OOF split + load encoder

In [ ]:
# Reuse Run 2 cache (mmap); mini-OOF split. Load pretrained encoder weights.
import torch
NPZ=os.path.join(CACHE_DIR,'m7_run2_data.npz')
def _mmap_member(name):
    out=os.path.join(CACHE_DIR,name+'.npy')
    if not os.path.exists(out):
        with zipfile.ZipFile(NPZ) as zf, zf.open(name+'.npy') as srcf, open(out,'wb') as dst:
            shutil.copyfileobj(srcf,dst,length=16*1024*1024)
    return np.load(out,mmap_mode='r')
X18=_mmap_member('X18')
z=np.load(NPZ,allow_pickle=True); fold18=z['fold18']; y18=z['y18']; z.close()
tr_all=np.where(np.isin(fold18,TRAIN_FOLDS))[0]; te_idx=np.where(np.isin(fold18,TEST_FOLDS))[0]
yte=y18[te_idx]; Xte=np.ascontiguousarray(X18[te_idx])
ENC_STATE=torch.load(ENC_PATH)
print('mini-OOF | train %d (%d WPW) | test %d (%d WPW) | encoder keys %d'%(
    len(tr_all),int(y18[tr_all].sum()),len(te_idx),int(yte.sum()),len(ENC_STATE)))
print('Block 9.1 OK.')


### Block 9.2: ResNet1d (transfer target)

In [ ]:
import torch.nn as nn
class BasicBlock1d(nn.Module):
    def __init__(self,cin,cout,stride=1):
        super().__init__()
        self.conv1=nn.Conv1d(cin,cout,7,stride=stride,padding=3,bias=False); self.bn1=nn.BatchNorm1d(cout)
        self.conv2=nn.Conv1d(cout,cout,7,padding=3,bias=False); self.bn2=nn.BatchNorm1d(cout)
        self.relu=nn.ReLU(inplace=True); self.down=None
        if stride!=1 or cin!=cout:
            self.down=nn.Sequential(nn.Conv1d(cin,cout,1,stride=stride,bias=False),nn.BatchNorm1d(cout))
    def forward(self,x):
        idt=x if self.down is None else self.down(x)
        o=self.relu(self.bn1(self.conv1(x))); o=self.bn2(self.conv2(o)); return self.relu(o+idt)
class ResNet1d(nn.Module):
    def __init__(self,ch=(16,32,64),pdrop=PDROP):
        super().__init__()
        self.stem=nn.Sequential(nn.Conv1d(12,ch[0],15,stride=2,padding=7,bias=False),
                                nn.BatchNorm1d(ch[0]),nn.ReLU(inplace=True),nn.MaxPool1d(4))
        self.layer1=BasicBlock1d(ch[0],ch[0],1); self.layer2=BasicBlock1d(ch[0],ch[1],2); self.layer3=BasicBlock1d(ch[1],ch[2],2)
        self.pool=nn.AdaptiveMaxPool1d(1); self.head=nn.Sequential(nn.Flatten(),nn.Dropout(pdrop),nn.Linear(ch[2],1))
        self.enc_mods=[self.stem,self.layer1,self.layer2,self.layer3]
    def forward(self,x):
        return self.head(self.pool(self.layer3(self.layer2(self.layer1(self.stem(x)))))).squeeze(1)
torch.set_num_threads(N_JOBS)
print('Block 9.2 OK | params:', sum(p.numel() for p in ResNet1d().parameters()))


### Block 9.3: Strong aug + train (scratch/frozen/finetune)

In [ ]:
# Strong aug + focal; train_one_transfer with mode = scratch | frozen | finetune.
import torch.nn.functional as Fnn
def focal_loss(logits,targets,gamma=FOCAL_GAMMA,alpha=FOCAL_ALPHA):
    ce=Fnn.binary_cross_entropy_with_logits(logits,targets,reduction='none')
    p=torch.sigmoid(logits); pt=torch.where(targets==1,p,1-p)
    a=torch.where(targets==1,torch.full_like(targets,alpha),torch.full_like(targets,1-alpha))
    return (a*(1-pt).pow(gamma)*ce).mean()
def augment(xb):   # strong
    T=xb.shape[2]; P=dict(shift=0.08,scale=0.3,noise=0.03,wander=0.08,ldrop=0.45)
    sh=max(1,int(P['shift']*T)); xb=torch.roll(xb,int(torch.randint(-sh,sh+1,(1,)).item()),dims=2)
    xb=xb*((1.0-P['scale'])+2*P['scale']*torch.rand(xb.shape[0],1,1)); xb=xb+P['noise']*torch.randn_like(xb)
    t=torch.linspace(0,1,T).view(1,1,-1); fb=0.5+2.0*torch.rand(xb.shape[0],1,1); ph=2*np.pi*torch.rand(xb.shape[0],1,1)
    xb=xb+P['wander']*torch.sin(2*np.pi*fb*t+ph)
    if torch.rand(1).item()<P['ldrop']: xb[:,int(torch.randint(0,12,(1,)).item()),:]=0.0
    if torch.rand(1).item()<0.25: xb[:,int(torch.randint(0,12,(1,)).item()),:]=0.0
    return xb
def predict(model,X):
    model.eval(); out=[]
    with torch.no_grad():
        for i in range(0,len(X),256):
            out.append(torch.sigmoid(model(torch.tensor(np.ascontiguousarray(X[i:i+256],dtype=np.float32)))).numpy())
    return np.nan_to_num(np.concatenate(out))
def set_train_mode(model,mode):
    model.train()
    if mode=='frozen':
        for m in model.enc_mods: m.eval()      # frozen encoder: keep pretrained BN stats, no dropout drift
def train_one_transfer(seed,Xtr,ytr,Xva,yva,mode):
    torch.manual_seed(seed); np.random.seed(seed); rng=np.random.default_rng(seed)
    model=ResNet1d()
    if mode!='scratch': model.load_state_dict(ENC_STATE,strict=False)         # transfer encoder convs
    if mode=='frozen':
        for mm in model.enc_mods:
            for p in mm.parameters(): p.requires_grad=False
    params=[p for p in model.parameters() if p.requires_grad]
    opt=torch.optim.Adam(params,lr=LR,weight_decay=WD)
    pos=np.where(ytr==1)[0]; neg=np.where(ytr==0)[0]; half=BATCH//2
    Xv=torch.tensor(np.ascontiguousarray(Xva,dtype=np.float32)); yv=torch.tensor(yva)
    steps=max(1,len(ytr)//BATCH); best=1e9; best_state=None; bad=0
    for ep in range(EPOCHS):
        set_train_mode(model,mode)
        for _ in range(steps):
            bi=np.concatenate([rng.choice(pos,half,True),rng.choice(neg,half,True)])
            xb=augment(torch.tensor(np.ascontiguousarray(Xtr[bi],dtype=np.float32))); yb=torch.tensor(ytr[bi])
            opt.zero_grad(); loss=focal_loss(model(xb),yb); loss.backward()
            torch.nn.utils.clip_grad_norm_(params,5.0); opt.step()
        model.eval()
        with torch.no_grad():
            vl=[focal_loss(model(Xv[i:i+256]),yv[i:i+256]).item() for i in range(0,len(Xv),256)]
        vloss=float(np.mean(vl))
        if vloss<best-1e-4: best=vloss; best_state={k:v.clone() for k,v in model.state_dict().items()}; bad=0
        else: bad+=1
        if bad>=PATIENCE: break
    model.load_state_dict(best_state); return model
print('Block 9.3 OK.')


### Block 9.4: A/B loop (checkpointed per mode+seed)

In [ ]:
def build_pool(seed0):
    rng=np.random.default_rng(seed0)
    pos=tr_all[y18[tr_all]==1]; negall=tr_all[y18[tr_all]==0]
    neg=rng.choice(negall,min(N_NEG_TRAIN,len(negall)),replace=False)
    vp=pos.copy(); vn=neg.copy(); rng.shuffle(vp); rng.shuffle(vn)
    nvp=max(6,int(VAL_FRAC*len(vp))); nvn=int(VAL_FRAC*len(vn))
    val=np.concatenate([vp[:nvp],vn[:nvn]]); pool=np.concatenate([pos,neg]); tr=np.setdiff1d(pool,val)
    return tr,val
tr_idx,val_idx=build_pool(2024)
Xtr=np.ascontiguousarray(X18[tr_idx],dtype=np.float32); ytr=y18[tr_idx]; Xva=X18[val_idx]; yva=y18[val_idx]
print('pool: train %d (%d WPW) | val %d (%d WPW)'%(len(tr_idx),int(ytr.sum()),len(val_idx),int(yva.sum())))
CK=os.path.join(MODELS,'run9_ab_ckpt.npz'); store={}
if os.path.exists(CK): store=np.load(CK,allow_pickle=True)['store'].item(); print('[ckpt] resumed:',list(store.keys()))
rows=[]; t0=time.time()
for mode in MODES:
    seed_pred=[np.asarray(p) for p in store.get(mode,{}).get('pred',[])]; done=store.get(mode,{}).get('done',0)
    for si,s in enumerate(SEEDS):
        if si<done: continue
        model=train_one_transfer(s,Xtr,ytr,Xva,yva,mode)
        seed_pred.append(predict(model,Xte))
        store[mode]={'pred':[p.tolist() for p in seed_pred],'done':si+1}; np.savez(CK,store=np.array(store,dtype=object))
    ens=np.mean(seed_pred,0); ap=float(average_precision_score(yte,ens)); auc=float(roc_auc_score(yte,ens))
    lo,hi=_boot_ci(yte.astype(int),ens,average_precision_score); seed_ap=[float(average_precision_score(yte,p)) for p in seed_pred]
    rows.append(dict(mode=mode,AP=ap,AP_lo=lo,AP_hi=hi,AUC=auc,AP_seed_mean=float(np.mean(seed_ap)),AP_seed_std=float(np.std(seed_ap))))
    print('  [%-8s] AP %.3f CI[%.3f,%.3f] | per-seed %.3f+/-%.3f | %.1f min'%(mode,ap,lo,hi,np.mean(seed_ap),np.std(seed_ap),(time.time()-t0)/60))
ab=pd.DataFrame(rows)
print('Block 9.4 OK.'); print(ab.to_string(index=False))


### Block 9.5: Sub-gate decision

In [ ]:
# SUB-GATE: pretrained (best of frozen/finetune) must beat scratch by >=3 sigma (per-seed) AND not less stable.
base=ab[ab['mode']==BASELINE].iloc[0]; pre=ab[ab['mode']!=BASELINE].sort_values('AP_seed_mean',ascending=False).iloc[0]
def se(std): return std/np.sqrt(len(SEEDS))
sigma=np.sqrt(se(base.AP_seed_std)**2+se(pre.AP_seed_std)**2); delta=pre.AP_seed_mean-base.AP_seed_mean
gate=bool(delta>3*sigma and pre.AP_seed_std<=base.AP_seed_std)
print('=== M7 Run 9 - transfer A/B (mini-OOF, test folds 1-4) ===')
print(ab.sort_values('AP_seed_mean',ascending=False).to_string(index=False))
print('  scratch per-seed %.3f+/-%.3f | best pretrained (%s) %.3f+/-%.3f'%(base.AP_seed_mean,base.AP_seed_std,pre['mode'],pre.AP_seed_mean,pre.AP_seed_std))
print('  delta %+.3f (~%.1f sigma)'%(delta,delta/max(sigma,1e-9)))
print('  SUB-GATE (pretrained beats scratch >=3 sigma AND not less stable) : %s'%gate)
print('  -> %s'%('PASS: proceed to Run 10 (full OOF pretrained = M7-v2)' if gate else 'FAIL: STOP. Pretraining does not transfer -> M7-v1 stays final (strong paper negative).'))
import json,os
json.dump({'modes':ab.to_dict('records'),'scratch':float(base.AP_seed_mean),'best_pretrained':pre['mode'],
           'delta':float(delta),'delta_sigma':float(delta/max(sigma,1e-9)),'gate_pass':gate},
          open(os.path.join(METRICS,'m7_run9_transfer_ab.json'),'w'),indent=2,default=float)
print('Block 9.5 OK.')